# Loading Monthly Data to a Database

While it may seem unnecessary, one reason this is potentially helpful is the total size of the data.
It might be prohibitive to actually load everything into memory and as such it may be a better idea to upload data to a database and download specific parts as necessary for analysis.

In [3]:
import pandas as pd
import numpy as np
import json
import pymongo
import psycopg
import sqlalchemy

The relational schema will keep numeric data as-is but link them to their value labels through foreign keys.
Retrieve value labels from MongoDB.

In [4]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_var_labs = lfs_db['variable_labels']
varlabs = lfs_var_labs.find_one()['labels']
lfs_val_labs = lfs_db['value_labels']
vallabs = lfs_val_labs.find_one()['labels']
client.close()

Pinged your deployment. You successfully connected to MongoDB!


Before accessing the relational database, a few things will be helpful.
First is the `MetaData` object.

In [33]:
lfs_metadata = sqlalchemy.MetaData()

Second is a function to create a `Table` given a `DataFrame`.

In [129]:
def df_to_table(df:pd.DataFrame, id_cols:list[str], metadata:sqlalchemy.MetaData, table_name:str, for_ids:dict=None):
    meta_pd = df.convert_dtypes(convert_integer=True).dtypes
    meta_pd = meta_pd.map({
        pd.Int64Dtype(): sqlalchemy.Integer,
        pd.Float64Dtype(): sqlalchemy.Float,
        pd.StringDtype(): sqlalchemy.String,
        np.datetime64: sqlalchemy.DateTime
    })
    meta_sql = sqlalchemy.Table(table_name, metadata)
    for v, t in zip(meta_pd.index, meta_pd):
        if v in id_cols:
            meta_sql.append_column(column=sqlalchemy.Column(v, t, primary_key=True))
        elif for_ids is not None and v in for_ids:
            meta_sql.append_column(column=sqlalchemy.Column(v, t, sqlalchemy.ForeignKey(for_ids[v])))
        else:
            meta_sql.append_column(column=sqlalchemy.Column(v, t))
    return meta_sql

Test the function

In [104]:
df = pd.DataFrame.from_dict({'label': {i: s for s, i in vallabs[next(iter(vallabs))].items()}}).rename_axis(index='value').reset_index(drop=False)
df_to_table(df, ['value'], lfs_metadata, next(iter(vallabs)))

Table('SURVMNTH', MetaData(), Column('value', Integer(), table=<SURVMNTH>, primary_key=True, nullable=False), Column('label', String(), table=<SURVMNTH>), schema=None)

Keeping that example table in memory will be troublesome.

In [135]:
lfs_metadata.clear()

Testing with a LFS data set.

In [136]:
df = pd.read_csv('data/lfs-2022-01.tab', sep='\t')
df['year'] = df['SURVYEAR']
df['month'] = df['SURVMNTH']
df['day'] = 1
df['date'] = pd.to_datetime(df[['year','month','day']])
df = df.drop(columns=['year','month','day'])
df = df.rename(columns=dict(zip(df.columns, df.columns.str.lower())))